# Step 5.h — Climate × soil suitability on ERA5, global (EC-Earth3 grid): validation and binary threshold

**Combined suitability = climate probability × soil rating**, per crop and GAEZ input level (low, high):
- **climate**: `suit_proba` of 4.g on ERA5 1981–2014 (SVM probability at equal class priors, 0 where the climate is
  novel: MESS < 0 or NT2 > 1);
- **soil**: SSR of 2.g (HWSD v2.01 × GAEZ v5 Module IV, rainfed, area-weighted mean over the cell, permafrost = 0),
  divided by 100. Land cells with no soil rating at all (no HWSD soil unit: ice, open water) get 0.

The soil rating is applied as a multiplicative factor on the climate result, as GAEZ v5 Module V applies the soil
quality rating to the agro-climatic suitability (not a pass / fail mask). The product stays continuous (0–1).

**Binary map, two thresholds.** The main result is the continuous product (expected areas, below); the binary map
needs a cut-off, chosen on ERA5 (the only climate with an observed crop distribution) and applied unchanged to the
four AMOC scenarios in 6.b, so that a difference between scenarios comes from the climate, not from the threshold.
The score has no land use in it: a cell outside the cropland with the same score as a cultivated cell is as
suitable, so the threshold applies to all land (novel climates excepted). The presences are reliable (the crop
grows there), the absences are not (a cropland cell without the crop is often a choice of market or rotation):
1. **TSS max**: the score that maximises the true skill statistic (TSS = sensitivity + specificity − 1; Allouche et
   al. 2006, *J. Appl. Ecol.* 43, 1223–1232) on the training cells of 3.g (MIRCA2000 presences vs cropland absences).
   The "max SSS" threshold is the same with pseudo-absences as with true absences (Liu, White & Newell 2013,
   *J. Biogeogr.* 40, 778–789; Liu et al. 2005, *Ecography* 28, 385–393);
2. **presence only**: the highest score whose cells hold `KEEP_AREA` (95 %) of the MIRCA2000 harvested area of the
   crop (fixed sensitivity; Liu et al. 2005; Pearson et al. 2007, *J. Biogeogr.* 34, 102–117). No absence used.

The same rules for every layer (climate alone, combined, GAEZ v5), each at its own thresholds. The climate
probability on the training cells is in-sample: the thresholds are cut-off choices, the TSS values are optimistic
(the cross-validation of 3.g is the skill).

**Checks**:
1. **MIRCA2000**: AUC on the training cells (does soil add ranking information to climate?), share of the MIRCA2000
   harvested area of the crop in suitable cells;
2. **GAEZ v5** (SIX, HP8100 = 1981–2000, AgERA5, rainfed, high input HRLM / low input LRLM; wheat not split into
   winter / spring; rice = wetland rice RCW, as the soil table of 2.c): class mid-points (SI > 85 → 0.925 …
   SI > 0 → 0.05, not suitable → 0, water / no data left out) averaged over the cell (cos-latitude weights);
   correlation with the combined map and the same AUC / TSS / area as ours. GAEZ also includes terrain, ours not;
3. **too cold for the crop** (3.h): suitable cells below the FAO ECOCROP absolute minimum temperature, by region
   (MIRCA cropland, outside it, north of 50° N and Tibetan plateau outside it, Antarctica). Decides whether the
   ECOCROP filter is needed once soil is applied (6.6 of the notes).

**Areas**: Mha of land (cell area × land fraction of 0.1) suitable at the threshold, and the *expected area*
Σ score × land area (no threshold, the continuous product).

Outputs in `<out_root>/5_climate_soil/<route>/<variant>_<radiation>/`: `climate_soil_<crop>.nc`,
`validation_<set>.csv`, `thresholds_<set>.csv` (read by 6.b), `areas_<set>.csv`, `gaez_comparison_<set>.csv`, maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr, rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the set of 4.g
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")   # as 4.g (rad_era5 once the rsds of the hosing runs arrives);
                                                     # set for one run with the environment (CHOSEN_RAD=rad_era5)
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
LEVELS = ["low", "high"]           # GAEZ input level of the soil rating (2.g) and of GAEZ v5
GAEZ_CODE = {"Maize_rf": "MZE", "Potato_rf": "WPO", "Soybean_rf": "SOY", "Rice_rf": "RCW",
             "Wheat_rf": "WHE", "Wheat-winter_rf": "WHE", "Wheat-spring_rf": "WHE"}
GAEZ_WSIM = {"high": "HRLM", "low": "LRLM"}   # rainfed, high / low input
KEEP_AREA = 0.95                   # presence-only threshold: share of the MIRCA2000 harvested area kept suitable
CRITERIA = ["TSS max", f"{100 * KEEP_AREA:g} % of MIRCA area"]
# FAO ECOCROP absolute minimum temperature (degC) and the temperature compared, as 3.h
ECOCROP = {"Maize_rf": (10, "season"), "Potato_rf": (7, "season"), "Soybean_rf": (10, "season"), "Rice_rf": (10, "season"),
           "Wheat-spring_rf": (5, "season"), "Wheat-winter_rf": (5, "phase2"), "Wheat_rf": (5, "phase2")}

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
CLIM_DIR = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
GAEZ_DIR = cu.data_dir(CFG, "gaez_v5_hist")
OUT = cu.out_dir(CFG, "5_climate_soil", ROUTE, SET)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
CRO_ANY = cu.mirca_cropland(CFG).values                 # any MIRCA2000 cropland (no threshold), as 3.h
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
REGIONS = {"MIRCA cropland": LAND & CRO_ANY,
           "outside cropland": LAND & ~CRO_ANY,
           "north of 50 N, outside cropland": LAND & ~CRO_ANY & (LAT2 > 50),
           "Tibetan plateau, outside cropland": LAND & ~CRO_ANY & (LAT2 > 27) & (LAT2 < 40) & (LON2 > 75) & (LON2 < 105),
           "Antarctica": LAND & (LAT2 < -60)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
GAEZ_FILE = {(lv, c): GAEZ_DIR / f"GAEZ-V5.RES05-SIX.HP8100.AGERA5.HIST.{GAEZ_CODE[c]}.{GAEZ_WSIM[lv]}.tif"
             for lv in LEVELS for c in CROPS}
missing = sorted({p.name for p in GAEZ_FILE.values() if not p.exists()})
print(f"set {SET} | climate {CLIM_DIR} | outputs {OUT}\nGAEZ v5 files missing (comparison skipped): {missing or 'none'}")

## 1. Combined suitability, and the layers on the training cells

In [ ]:
SI_MID = np.array([np.nan, 0.925, 0.775, 0.625, 0.475, 0.325, 0.175, 0.05, 0.0, np.nan])   # GAEZ SIX class -> SI mid-point
_IDX = {}                                                                                 # (0 no data, 8 not suitable, 9 water)


def gaez_on_grid(path):
    # GAEZ v5 SIX (5 arc-min) -> mean SI mid-point of the pixels of each EC-Earth3 cell, cos-latitude weights
    with rasterio.open(path) as src:
        a, T = src.read(1), src.transform
    key = (a.shape, tuple(T))
    if key not in _IDX:
        lat = T.f + T.e * (np.arange(a.shape[0]) + 0.5); lon = T.c + T.a * (np.arange(a.shape[1]) + 0.5)
        I, _ = cu.cell_of(lat, np.zeros_like(lat), GRID); _, J = cu.cell_of(np.zeros_like(lon), lon, GRID)
        _IDX[key] = ((I[:, None] * GRID["lon"].size + J[None, :]).ravel(), np.repeat(np.cos(np.radians(lat)), a.shape[1]))
    flat, w = _IDX[key]
    s = SI_MID[np.minimum(a, 9)].ravel()
    ok = np.isfinite(s)
    num = np.bincount(flat[ok], weights=s[ok] * w[ok], minlength=LAND.size)
    den = np.bincount(flat[ok], weights=w[ok], minlength=LAND.size)
    with np.errstate(invalid="ignore", divide="ignore"):
        g = np.where(den > 0, num / den, np.nan).reshape(LAND.shape)
    return np.where(LAND, g, np.nan)


mha = lambda mask: float(np.nansum(LAND_MHA * mask))
val_rows, thr_rows, area_rows, gaez_rows, MAPS = [], [], [], [], {}
for crop in CROPS:
    with xr.open_dataset(CLIM_DIR / f"suitability_{crop}.nc") as ds:
        clim = ds["suit_proba"].sel(scenario="ERA5").values.astype(float)
        novel = ds["novel"].sel(scenario="ERA5").values == 1
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv", usecols=["lat", "lon", "label"])
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    y = tr.label.values
    tmin, which = ECOCROP[crop]
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        T = cu.growing_temperature(ds, which)
    cold = LAND & np.isfinite(T) & (T < tmin)
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)

    layers, soil = {"climate": clim}, {}
    for lv in LEVELS:
        s = np.where(LAND, SOIL[lv][crop].values / 100.0, np.nan)
        soil[lv] = s
        layers[f"soil ({lv})"] = s
        layers[f"combined ({lv})"] = np.where(LAND & np.isfinite(clim), clim * np.nan_to_num(s, nan=0.0), np.nan)
        if GAEZ_FILE[(lv, crop)].exists():
            layers[f"GAEZ v5 ({lv})"] = gaez_on_grid(GAEZ_FILE[(lv, crop)])
    thr = {}                                               # (layer, criterion) -> threshold
    for name, a in layers.items():
        v = a[ii, jj]; ok = np.isfinite(v)
        val_rows.append({"crop": crop, "layer": name, "training cells": int(ok.sum()), "AUC": roc_auc_score(y[ok], v[ok]),
                         "mean on presences": v[ok][y[ok] == 1].mean(), "mean on absences": v[ok][y[ok] == 0].mean()})
        if name.startswith("soil"):
            continue                                       # soil alone: a rating, not a suitability map
        thr[(name, CRITERIA[0])] = cu.tss_threshold(v, y)[0]
        thr[(name, CRITERIA[1])] = cu.area_threshold(a, crop_ha, KEEP_AREA)
        for crit in CRITERIA:
            t = thr[(name, crit)]
            sens, spec = (v[ok][y[ok] == 1] >= t).mean(), (v[ok][y[ok] == 0] < t).mean()
            thr_rows.append({"crop": crop, "layer": name, "criterion": crit, "threshold": t, "TSS": sens + spec - 1,
                             "sensitivity": sens, "specificity": spec})
    thr[("climate", "p >= 0.5 (4.g)")] = 0.5
    for (name, crit), t in thr.items():
        a = layers[name]
        suit = np.nan_to_num(a) >= t
        r = {"crop": crop, "layer": name, "criterion": crit, "threshold": t, "suitable land (Mha)": mha(suit)}
        for reg, rm in REGIONS.items():
            r[f"{reg} (Mha)"] = mha(suit & rm)
        r["below TMIN, outside cropland (Mha)"] = mha(suit & cold & REGIONS["outside cropland"])
        r["below TMIN, north of 50 N outside cropland (Mha)"] = mha(suit & cold & REGIONS["north of 50 N, outside cropland"])
        r["MIRCA area of the crop in suitable cells (%)"] = 100 * crop_ha[suit].sum() / crop_ha.sum()
        r["expected area, no threshold (Mha)"] = float(np.nansum(np.nan_to_num(a) * LAND_MHA))
        r["mean score below TMIN, north of 50 N outside cropland"] = float(np.nanmean(a[cold & REGIONS["north of 50 N, outside cropland"]]))
        area_rows.append(r)
    for lv in LEVELS:
        g, c = layers.get(f"GAEZ v5 ({lv})"), layers[f"combined ({lv})"]
        if g is None:
            continue
        both = np.isfinite(g) & np.isfinite(c)
        r = {"crop": crop, "level": lv, "cells": int(both.sum()), "Pearson r": np.corrcoef(c[both], g[both])[0, 1],
             "Spearman rho": spearmanr(c[both], g[both])[0],
             "Pearson r, MIRCA cropland": np.corrcoef(c[both & CRO_ANY], g[both & CRO_ANY])[0, 1]}
        for crit in CRITERIA:                              # each map at its own threshold of the same criterion
            sg, sc = g >= thr[(f"GAEZ v5 ({lv})", crit)], c >= thr[(f"combined ({lv})", crit)]
            r.update({f"both suitable, {crit} (Mha)": mha(both & sg & sc), f"ours only, {crit} (Mha)": mha(both & sc & ~sg),
                      f"GAEZ only, {crit} (Mha)": mha(both & sg & ~sc)})
        gaez_rows.append(r)
    no_soil = LAND & ~np.isfinite(soil[LEVELS[0]])
    MAPS[crop] = {"layers": layers, "thr": thr, "cold": cold}
    dv = {"climate": (("lat", "lon"), clim.astype("float32"), {"long_name": "climate suitability probability (4.g, ERA5), 0 where novel"}),
          "novel": (("lat", "lon"), novel.astype("int8"), {"long_name": "novel climate (MESS < 0 or NT2 > 1)"}),
          "below_tmin": (("lat", "lon"), cold.astype("int8"), {"long_name": f"growing-period temperature ({which}) below the ECOCROP TMIN {tmin} degC"})}
    for lv in LEVELS:
        c = layers[f"combined ({lv})"]
        dv[f"soil_{lv}"] = (("lat", "lon"), soil[lv].astype("float32"), {"long_name": f"soil suitability rating / 100 (2.g), {lv} input"})
        dv[f"combined_{lv}"] = (("lat", "lon"), c.astype("float32"), {"long_name": f"climate x soil suitability, {lv} input",
                                                                        "note": "0 on land cells without a soil rating"})
        for crit, key in zip(CRITERIA, ("tss", "area")):
            t = thr[(f"combined ({lv})", crit)]
            dv[f"suitable_{key}_{lv}"] = (("lat", "lon"), np.where(np.isfinite(c), c >= t, np.nan).astype("float32"),
                                          {"long_name": f"binary climate x soil suitability, {lv} input, threshold {crit}",
                                           "threshold": t})
        if f"GAEZ v5 ({lv})" in layers:
            dv[f"gaez_v5_{lv}"] = (("lat", "lon"), layers[f"GAEZ v5 ({lv})"].astype("float32"),
                                   {"long_name": f"GAEZ v5 SIX HP8100 {GAEZ_CODE[crop]} {GAEZ_WSIM[lv]}, mean class mid-point"})
    xr.Dataset(dv, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "set": SET, "route": ROUTE, "climate": f"{ERA5_NAME} {ERA5_PERIOD[0]}-{ERA5_PERIOD[1]}"}
               ).to_netcdf(OUT / f"climate_soil_{crop}.nc")
    print(f"{crop}: land without soil rating {mha(no_soil):.0f} Mha, combined thresholds "
          + ", ".join(f"{lv} {thr[(f'combined ({lv})', k)]:.2f} ({k})" for lv in LEVELS for k in CRITERIA), flush=True)

## 2. Results

(1) AUC on the training cells: soil alone, climate alone, combined, GAEZ v5. (2) TSS thresholds. (3) Areas at the
threshold and expected areas; suitable cells below the ECOCROP TMIN. (4) Agreement with GAEZ v5.

In [ ]:
val = pd.DataFrame(val_rows).set_index(["crop", "layer"])
thr_tab = pd.DataFrame(thr_rows).set_index(["crop", "layer", "criterion"])
areas = pd.DataFrame(area_rows).set_index(["crop", "layer", "criterion"])
gaez = pd.DataFrame(gaez_rows).set_index(["crop", "level"]) if gaez_rows else pd.DataFrame()
val.to_csv(OUT / f"validation_{SET}.csv"); thr_tab.to_csv(OUT / f"thresholds_{SET}.csv")
areas.to_csv(OUT / f"areas_{SET}.csv"); gaez.to_csv(OUT / f"gaez_comparison_{SET}.csv")
with pd.option_context("display.max_rows", 300, "display.max_columns", 30, "display.width", 250):
    print("(1) AUC on the training cells (MIRCA2000 presences vs cropland absences, ERA5)")
    display(val["AUC"].unstack("layer").round(3))
    print("(2) thresholds, and their TSS / sensitivity / specificity on the training cells")
    display(thr_tab.round(3))
    print("(3) areas (Mha) at the threshold of each layer")
    display(areas.round(1).assign(threshold=areas.threshold.round(3)))
    print("(4) combined vs GAEZ v5")
    display(gaez.round(3))

## 3. Maps

In [ ]:
cls_cmap = ListedColormap(["#e6e6e6", "#4daf4a", "#e41a1c"])   # not suitable | suitable | suitable but below TMIN
for crop in CROPS:
    L, thr, cold = MAPS[crop]["layers"], MAPS[crop]["thr"], MAPS[crop]["cold"]
    top = ["climate"] + [f"combined ({lv})" for lv in LEVELS] + [k for k in (f"GAEZ v5 ({lv})" for lv in LEVELS[::-1]) if k in L][:1]
    fig, axes = plt.subplots(2, 4, figsize=(26, 7))
    for ax, k in zip(axes[0], top):
        im = ax.pcolormesh(GRID["lon"], GRID["lat"], L[k], vmin=0, vmax=1, cmap="YlGn", shading="nearest")
        ax.set_title(f"{crop}: {k}", fontsize=9)
    plt.colorbar(im, ax=axes[0].tolist(), shrink=0.8)
    for ax, (lv, crit) in zip(axes[1], [(lv, crit) for crit in CRITERIA for lv in LEVELS]):
        k = f"combined ({lv})"
        t = thr[(k, crit)]
        s = np.nan_to_num(L[k]) >= t
        c = np.where(LAND, np.where(s & cold, 2, s.astype(float)), np.nan)
        ax.pcolormesh(GRID["lon"], GRID["lat"], c, vmin=-0.5, vmax=2.5, cmap=cls_cmap, shading="nearest")
        ax.set_title(f"{k} >= {t:.2f} ({crit}); red: below TMIN", fontsize=9)
    plt.savefig(OUT / f"climate_soil_maps_{SET}_{crop}.png", dpi=80, bbox_inches="tight"); plt.show()